# Cleaning the data

Most of my data is already in the format I expect it to be in since it's manually gathered - but my process is first downloading from different tabs of a Google Sheets file, so some running it through some reusable processes here can help prepare for the later modeling.

As I continue to collect more data on more artists, the following process will be used to prep the data and keep everything organized.

In [39]:
import pandas as pd
import re
import json
import os

### 1. Download the updated data
Downloading from Google Sheets which gives me CSV files in this format: "DATA - dataset_name.csv"

### 2. Rename and organize files
Place in a directory corresponding to updated date so I can see which version of the data I'm working with as I collect more

In [40]:
today = pd.to_datetime('today').strftime('%m-%d')

In [43]:
def process_files():
    unprocessed_files = pd.Series(os.listdir('../../data/downloaded/unprocessed'))
    for file in unprocessed_files:
        print(f'Processing file: {file}')
        if "DATA" in file:
            df = pd.read_csv(f'../../data/downloaded/unprocessed/{file}')
            
            if not os.path.exists(f'../../data/downloaded/processed/{today}'):
                os.makedirs(f'../../data/downloaded/processed/{today}')
            cleaned_file_name = file.split(" - ")[1].split(".")[0]
            new_file_name = f'{cleaned_file_name}.csv'

            df.to_csv(f'../../data/downloaded/processed/{today}/{new_file_name}', index=False)


** Note: ignoring `institutions.csv` file since this is the only constant as I collect more data, it can live in the `constants` folder

In [44]:
process_files()

Processing file: DATA - artworks.csv
Processing file: .DS_Store
Processing file: DATA - artist_words.csv
Processing file: DATA - exhibitions.csv
Processing file: DATA - artists.csv


### 3. Standardize mediums
I also need to extract the mediums to a more standardized format within the `artworks.csv` file - I have a working mediums file that will be a resource to check against and might continue to need to be updated

In [45]:
artworks_df = pd.read_csv(f'../../data/downloaded/processed/{today}/artworks.csv')
mediums_df = pd.read_csv('../../data/downloaded/constants/mediums.csv')

artworks_df.head()

,id,artist,title,alt_text,description,date_created,date_acquired_or_updated,institution,medium,image_url
0,W001,1,Invisible Ink,NaN,NaN,2010,1/27/2021,0,Two chromogenic prints | Photograph,https://www.moma.org/media/W1siZiIsIjQ5NzQ1NCJ...
1,W002,2,Oops!,NaN,NaN,2000,4/14/2009,0,"Three-channel video (color, sound) | Installat...",https://www.moma.org/media/W1siZiIsIjE1ODc1NyJ...
2,W003,2,Lucky Tiger #151,NaN,NaN,2009,5/12/2010,0,Chromogenic print with ink fingerprints | Phot...,https://www.moma.org/media/W1siZiIsIjIxNjQ4MiJ...
3,W004,2,Lucky Tiger #169,NaN,NaN,2009,5/12/2010,0,Chromogenic print with ink fingerprints | Phot...,https://www.moma.org/media/W1siZiIsIjIxNjQ4MyJ...
4,W005,2,Lucky Tiger #181,NaN,NaN,2009,5/12/2010,0,Chromogenic print with ink fingerprints | Phot...,https://www.moma.org/media/W1siZiIsIjIxNjQ4NCJ...


In [46]:
# manually created synonyms file based on previous medium extraction
synonyms_df = pd.read_csv('../../data/utils/medium_synonyms.csv')

synonyms_df.head()

,medium_id,term,priority
0,M06,sculpture,1
1,M04,painting,1
2,M03,drawing,1
3,M03,drawing and watercolor,1
4,M02,print,1


Normalize the text and build lookups before processing

In [47]:
def normalize(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).lower().strip())

def singularize_token(token):
    if len(token) > 3 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token

def canonicalize_text(text):
    text = normalize(text)
    text = re.sub(r"[&/+]+", " and ", text)
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return " ".join(singularize_token(tok) for tok in text.split())

artworks_df["medium_clean"] = artworks_df["medium"].apply(canonicalize_text)
synonyms_df["term"] = synonyms_df["term"].apply(canonicalize_text)
mediums_df["term"] = mediums_df["medium_title"].apply(canonicalize_text)

# Use both curated synonyms and canonical medium titles as lookup terms.
lookup_df = pd.concat(
    [
        synonyms_df[["medium_id", "term", "priority"]],
        mediums_df[["medium_id", "term"]].assign(priority=1),
    ],
    ignore_index=True,
).drop_duplicates()

priority_1 = lookup_df[lookup_df["priority"] == 1]
priority_2 = lookup_df[lookup_df["priority"] == 2]

p1_pairs = list(priority_1[["medium_id", "term"]].itertuples(index=False, name=None))
p2_pairs = list(priority_2[["medium_id", "term"]].itertuples(index=False, name=None))

In [48]:
def parse_medium(text):
    text = canonicalize_text(text)
    
    for mid, term in p1_pairs:
        if re.search(rf"\b{re.escape(term)}\b", text):
            return [mid]
    
    matches = []
    
    for mid, term in p2_pairs:
        if re.search(rf"\b{re.escape(term)}\b", text):
            matches.append(mid)
    
    return sorted(set(matches))

Parse the column, check result

In [49]:
artworks_df["mediums_parsed"] = artworks_df["medium"].apply(
    lambda x: json.dumps(parse_medium(x))
)

display(
    artworks_df[
        ["medium", "mediums_parsed"]
    ].head()
)

,medium,mediums_parsed
0,Two chromogenic prints | Photograph,"[""M02""]"
1,"Three-channel video (color, sound) | Installat...","[""M05""]"
2,Chromogenic print with ink fingerprints | Phot...,"[""M02""]"
3,Chromogenic print with ink fingerprints | Phot...,"[""M02""]"
4,Chromogenic print with ink fingerprints | Phot...,"[""M02""]"


For anything that didn't get matched - print the source text so the medium/lookup file can be updated

In [50]:
artworks_df["matched_count"] = artworks_df["mediums_parsed"].apply(lambda x: len(json.loads(x)))

unmatched_df = (
    artworks_df.loc[artworks_df["matched_count"] == 0, ["medium"]]
    .value_counts()
    .reset_index(name="count")
)

for _, row in unmatched_df.iterrows():
    print(f'- {row["medium"]} ({row["count"]})')

In [51]:
artworks_df.columns

Index(['id', 'artist', 'title', 'alt_text', 'description', 'date_created',
       'date_acquired_or_updated', 'institution', 'medium', 'image_url',
       'medium_clean', 'mediums_parsed', 'matched_count'],
      dtype='object')

Drop helper columns and re-export

In [52]:
artworks_df = artworks_df.drop(columns=["medium_clean", "matched_count"])
artworks_df.to_csv(f"../../data/downloaded/processed/{today}/artworks.csv", index=False)

### Examine words data

In [ ]:
# how much data for each artist (in rows/sources)
artist_wds = pd.read_csv(f"../../data/downloaded/processed/{today}/artist_words.csv")
artist_wds.groupby("artist").size()

artist
0     10
1      7
2      4
3     13
4      8
5      5
6      9
7      9
8      5
9      8
10     3
11     2
12     5
13     4
14     7
15     5
16     6
17     4
18     8
19     7
20     5
21     2
22     8
23     6
24     6
25     6
26     4
27     3
28     5
29     5
dtype: int64

In [54]:
# how many words per artist?
artist_wds["word_count"] = artist_wds["text"].apply(lambda x: len(str(x).split()))
artist_wds.groupby("artist")["word_count"].sum()

artist
0      2633
1      6216
2      6913
3      9641
4      9070
5      3394
6     11022
7      5150
8      5823
9      7451
10     9359
11     4865
12     5935
13     4526
14    38683
15     7633
16     7017
17      987
18     6759
19     8186
20     4425
21    15260
22     9702
23     3121
24     2750
25    21904
26     5840
27     1800
28     2264
29    14034
Name: word_count, dtype: int64